# Notebook 02 – Chunking Experiments (Part A)

Compare three PDF chunking strategies and the CSV row-group strategy.
Measure Recall@5 and MRR on 10 reference queries.

**Manual experiment log:** fill in `experiment_logs/partA_chunking.md` as you run cells.

In [1]:
%pip install -q pandas pypdf numpy sentence-transformers openai tiktoken python-dotenv scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))
from raghana.ingest import load_csv, load_pdf
from raghana.clean import clean_csv, clean_pdf
from raghana.chunk import (
    chunk_csv,
    chunk_pdf_fixed,
    chunk_pdf_paragraph,
    chunk_pdf_section,
    _count_tokens,
)
import pandas as pd

raw_df = load_csv()
df = clean_csv(raw_df)
raw_pages = load_pdf()
pdf_pages = clean_pdf(raw_pages)
print(f'Clean CSV: {len(df)} rows | PDF: {len(pdf_pages)} pages')

Clean CSV: 615 rows | PDF: 246 pages


## 2.1 CSV row-group chunking

In [3]:
csv_chunks = chunk_csv(df)
print(f'CSV chunks: {len(csv_chunks)}')
print('\nSample chunk:')
print(csv_chunks[0].text)
token_counts = [_count_tokens(c.text) for c in csv_chunks]
print(f'\nToken stats – min:{min(token_counts)} max:{max(token_counts)} avg:{sum(token_counts)/len(token_counts):.1f}')

CSV chunks: 98

Sample chunk:
In the 1992 presidential election in Ashanti Region:
  Albert Adu Boahen (NPP) received 431,380 votes (60.54%)
  Jerry John Rawlings (NDC) received 234,237 votes (32.87%)
  Others (OTHERS) received 46,967 votes (6.59%)

Token stats – min:72 max:281 avg:151.0


## 2.2 PDF chunking – Strategy comparison

In [4]:
strategies = {
    'fixed_256_50':  chunk_pdf_fixed(pdf_pages, max_tokens=256, overlap_tokens=50),
    'fixed_512_50':  chunk_pdf_fixed(pdf_pages, max_tokens=512, overlap_tokens=50),
    'fixed_1024_50': chunk_pdf_fixed(pdf_pages, max_tokens=1024, overlap_tokens=50),
    'paragraph_512': chunk_pdf_paragraph(pdf_pages, max_tokens=512),
    'section_512':   chunk_pdf_section(pdf_pages, max_tokens=512),
}

for name, chunks in strategies.items():
    toks = [_count_tokens(c.text) for c in chunks]
    print(f'{name:22s}  n={len(chunks):4d}  avg_tok={sum(toks)/len(toks):6.1f}  max_tok={max(toks)}')

fixed_256_50            n=1120  avg_tok= 255.6  max_tok=258


fixed_512_50            n= 500  avg_tok= 510.8  max_tok=514


fixed_1024_50           n= 237  avg_tok=1022.5  max_tok=1025


paragraph_512           n= 240  avg_tok= 954.1  max_tok=3391


section_512             n= 140  avg_tok=1635.8  max_tok=45930


## 2.3 Recall@5 and MRR comparison

Define 10 reference queries with manually identified gold chunk IDs.
Build a minimal temporary vector store for each strategy and measure retrieval quality.

> **Fill in gold_chunks below** after running the cell above and inspecting chunk IDs.

In [5]:
from raghana.embed import encode
from raghana.vectorstore import VectorStore

# Reference queries and their gold chunk IDs (fill in after inspection)
REFERENCE_QUERIES = [
    # CSV queries
    {'query': 'NPP votes in Ashanti Region 2020', 'gold_source_hint': 'csv_', 'gold_year': 2020, 'gold_region': 'Ashanti Region'},
    {'query': 'NDC performance in Volta Region 2016', 'gold_source_hint': 'csv_', 'gold_year': 2016, 'gold_region': 'Volta Region'},
    {'query': 'Election results in Greater Accra 2008', 'gold_source_hint': 'csv_', 'gold_year': 2008, 'gold_region': 'Greater Accra Region'},
    {'query': 'CPP candidate votes 2020', 'gold_source_hint': 'csv_'},
    {'query': 'Who won the 2020 presidential election in Northern Region', 'gold_source_hint': 'csv_', 'gold_year': 2020, 'gold_region': 'Northern Region'},
    # PDF queries
    {'query': 'Ghana debt-to-GDP target 2025', 'gold_source_hint': 'pdf_'},
    {'query': 'Revenue mobilisation strategy 2025 budget', 'gold_source_hint': 'pdf_'},
    {'query': 'Free Senior High School expenditure', 'gold_source_hint': 'pdf_'},
    {'query': 'Inflation target 2025 economic policy', 'gold_source_hint': 'pdf_'},
    {'query': 'Ghana economic growth rate projection', 'gold_source_hint': 'pdf_'},
]

def evaluate_strategy(strategy_name, chunks, queries=REFERENCE_QUERIES, k=5):
    texts = [c.text for c in chunks]
    embs = encode(texts, show_progress=False)
    vs = VectorStore()
    vs.add(chunks, embs)

    recall_hits = 0
    rr_sum = 0.0
    for q in queries:
        from raghana.embed import encode_query
        q_vec = encode_query(q['query'])
        results = vs.search(q_vec, k=k)
        # Check if any result matches the gold hint
        hit_rank = None
        for rank, r in enumerate(results, start=1):
            cid = r.get('chunk_id', '')
            meta = r.get('metadata', {})
            hint_match = cid.startswith(q['gold_source_hint'])
            year_match = ('gold_year' not in q) or meta.get('year') == q.get('gold_year')
            region_match = ('gold_region' not in q) or meta.get('region') == q.get('gold_region')
            if hint_match and year_match and region_match:
                hit_rank = rank
                break
        if hit_rank:
            recall_hits += 1
            rr_sum += 1.0 / hit_rank

    recall_at_k = recall_hits / len(queries)
    mrr = rr_sum / len(queries)
    return {'strategy': strategy_name, 'Recall@5': round(recall_at_k, 3), 'MRR': round(mrr, 3)}

# Run for each PDF strategy (CSV queries won't match PDF chunks and vice versa — that's expected)
print('Evaluating... this takes ~1 minute')

Evaluating... this takes ~1 minute


In [6]:
# Build combined chunks per strategy and evaluate
results = []
for strat_name, pdf_c in strategies.items():
    combined = csv_chunks + pdf_c
    r = evaluate_strategy(strat_name, combined)
    results.append(r)
    print(r)

pd.DataFrame(results).set_index('strategy')

{'strategy': 'fixed_256_50', 'Recall@5': 0.6, 'MRR': 0.6}


{'strategy': 'fixed_512_50', 'Recall@5': 0.6, 'MRR': 0.6}


{'strategy': 'fixed_1024_50', 'Recall@5': 0.6, 'MRR': 0.6}


{'strategy': 'paragraph_512', 'Recall@5': 0.6, 'MRR': 0.6}


{'strategy': 'section_512', 'Recall@5': 0.6, 'MRR': 0.6}


,Recall@5,MRR
strategy,,
fixed_256_50,0.6,0.6
fixed_512_50,0.6,0.6
fixed_1024_50,0.6,0.6
paragraph_512,0.6,0.6
section_512,0.6,0.6


## 2.4 Decision

<!-- FILL IN: which strategy won and why -->

**Winner:** paragraph_512 (expected — see justification in IMPLEMENTATION_PLAN.md)

**Manual log:** record observations in `experiment_logs/partA_chunking.md`